# 05 神经网络训练流程

前面我们已经知道：

```text
损失函数负责衡量错误
梯度下降负责更新参数
反向传播负责计算梯度
```

这一节把这些概念串成一个完整训练流程。重点不是代码，而是理解训练时到底发生了什么。

## 1. 训练神经网络到底在训练什么

神经网络里真正被训练的是参数，主要包括权重和偏置：

$$
\theta=\{\mathbf{W}_1,\mathbf{b}_1,\mathbf{W}_2,\mathbf{b}_2,\dots\}
$$

网络结构本身通常是我们提前设计好的，比如有几层、每层多少神经元、使用什么激活函数。

训练的目标是找到一组更好的参数，让模型预测更接近真实答案：

$$
\hat{y}=f_{\theta}(x)
$$

$$
\min_{\theta}\mathcal{L}(\hat{y},y)
$$

所以训练不是让网络“记住代码”，也不是让网络“自己长出结构”，而是在固定结构里不断调整参数。

## 2. 训练数据是什么

训练数据通常由很多样本组成。每个样本有输入和标签：

$$
(x^{(i)},y^{(i)})
$$

其中：

- $x^{(i)}$ 是第 $i$ 个样本的输入。
- $y^{(i)}$ 是第 $i$ 个样本的真实答案。

整个训练集可以写成：

$$
\mathcal{D}=\{(x^{(1)},y^{(1)}),(x^{(2)},y^{(2)}),\dots,(x^{(m)},y^{(m)})\}
$$

其中 $m$ 表示样本数量。

神经网络不是从一条样本里学出全部规律，而是从大量样本中反复调整参数，逐渐找到更稳定的模式。

## 3. 为什么不能只看训练集

如果模型只在训练集上表现好，不代表它真的学会了规律。

它可能只是把训练数据背下来了。

所以我们通常把数据分成三部分：

| 数据集 | 作用 |
|---|---|
| 训练集 | 用来更新参数 |
| 验证集 | 用来观察模型在没参与训练的数据上表现如何 |
| 测试集 | 用来在最终阶段评估模型泛化能力 |

训练集回答的是：模型能不能把见过的数据学好。

验证集回答的是：模型能不能在没直接训练过的数据上表现好。

测试集回答的是：模型最终能不能交卷。

## 4. 什么是泛化能力

泛化能力指的是：模型在没见过的新数据上表现好的能力。

训练神经网络不是为了让它只记住训练集，而是希望它学到背后的规律。

如果模型在训练集上损失很低，但在验证集上损失很高，说明它可能记住了训练集细节，却没有学到真正规律。

这就是过拟合的典型表现。

## 5. 什么是 batch

如果训练集有 $m$ 个样本，我们当然可以一次性把所有样本都送进模型。

但现实中数据可能很多，一次全部放进去会占用大量内存，而且每次更新参数都要等所有样本算完。

所以训练时通常把训练集切成很多小块，每一小块叫一个 batch。

假设 batch size 是 $B$，一个 batch 可以写成：

$$
\{(x^{(1)},y^{(1)}),\dots,(x^{(B)},y^{(B)})\}
$$

模型每次用一个 batch 计算损失、反向传播、更新参数。

## 6. batch size 太大或太小会怎样

batch size 太小，每次看到的数据少，梯度会比较不稳定。它像是在听很少几个人的意见就做决定，方向可能有点抖。

batch size 太大，每次看到的数据多，梯度更稳定，但计算和显存压力更大，而且每次参数更新之间等待更久。

可以粗略理解为：

| batch size | 特点 |
|---|---|
| 小 | 更新频繁，梯度噪声大，占用内存小 |
| 大 | 梯度稳定，占用内存大，更新次数少 |

所以 batch size 是训练效率、内存占用、梯度稳定性之间的折中。

## 7. 什么是 iteration

iteration 指一次参数更新。

一个 batch 进入模型，完成下面四步：

```text
前向传播 -> 计算损失 -> 反向传播 -> 更新参数
```

这就完成了一次 iteration。

如果训练集有 $1000$ 个样本，batch size 是 $100$，那么一个 epoch 里大约有：

$$
\frac{1000}{100}=10
$$

次 iteration。

## 8. 什么是 epoch

epoch 指模型完整看完一遍训练集。

也就是说：

$$
1 个 Epoch = 整个训练数据集中的所有样本，全部前向传播(前向计算)并反向传播(反向更新梯度)一次。
$$

如果训练集有 $m$ 个样本，模型把这 $m$ 个样本都用来训练过一次，就完成了 $1$ 个 epoch。

注意：epoch 不是一次参数更新。

一次 epoch 里通常包含很多次 iteration。

它们的关系是：

$$
\text{iterations per epoch}=\left\lceil\frac{m}{B}\right\rceil
$$

其中 $B$ 是 batch size。

## 9. 一次完整训练步骤

对一个 batch 来说，训练步骤是：

1. 取一批数据。

$$
(\mathbf{X},\mathbf{y})
$$

2. 前向传播，得到预测。

$$
\hat{\mathbf{y}}=f_{\theta}(\mathbf{X})
$$

3. 计算损失。

$$
\mathcal{L}=\mathcal{L}(\hat{\mathbf{y}},\mathbf{y})
$$

4. 反向传播，计算梯度。

$$
\nabla_{\theta}\mathcal{L}
$$

5. 使用优化器更新参数。

$$
\theta \leftarrow \theta-\eta\nabla_{\theta}\mathcal{L}
$$

这五步会对很多 batch 重复很多轮。

## 10. 优化器是什么

优化器负责根据梯度更新参数。

最朴素的优化器就是梯度下降：

$$
\theta \leftarrow \theta-\eta\nabla_{\theta}\mathcal{L}
$$

但实际训练中，还会有更复杂的优化器，比如 SGD、Momentum、Adam。

先不要急着记 API。先理解它们都在做同一件事：

```text
根据梯度，决定参数下一步怎么走
```

不同优化器的区别在于：走得是否更稳，是否利用历史梯度，是否给不同参数自适应调整步幅。

## 11. 什么是训练模式和评估模式

训练时，模型会更新参数。

评估时，模型只做预测，不更新参数。

这两种状态的目的不同：

| 阶段 | 是否更新参数 | 目的 |
|---|---|---|
| 训练 | 是 | 让模型变好 |
| 验证 / 测试 | 否 | 检查模型表现 |

为什么要分开？因为验证集和测试集是用来检查模型泛化能力的。如果你用它们更新参数，就相当于让模型偷看答案，评估就不客观了。

## 12. 什么是过拟合

过拟合指模型在训练集上表现很好，但在新数据上表现不好。

可以理解成：模型没有学到规律，而是把训练集细节背下来了。

典型现象是：

$$
\mathcal{L}_{train}\downarrow
$$

但：

$$
\mathcal{L}_{val}\uparrow
$$

也就是训练损失继续下降，验证损失反而上升。

这说明模型越来越会处理训练集，但越来越不会处理没见过的数据。

## 13. 什么是欠拟合

欠拟合指模型连训练集都学不好。

典型现象是：

$$
\mathcal{L}_{train}\text{ 很高}
$$

$$
\mathcal{L}_{val}\text{ 也很高}
$$

这通常说明模型能力不够、训练不充分、特征不合适，或者学习率等训练设置有问题。

欠拟合像是学生连课本例题都没学会。

过拟合像是学生只背熟了课本例题，但换一道题就不会。

## 14. 如何观察训练是否正常

训练时最常观察两类曲线：

1. loss 曲线
2. accuracy 曲线

loss 表示模型错得有多严重。

accuracy 表示分类任务中预测正确的比例。

如果训练正常，常见趋势是：

$$
\mathcal{L}_{train}\downarrow
$$

$$
\operatorname{accuracy}_{train}\uparrow
$$

同时验证集指标也应该在一定程度上变好。

如果训练 loss 完全不下降，可能是学习率、初始化、数据、模型结构或梯度传播出了问题。

如果训练 loss 下降，但验证 loss 上升，可能是过拟合。

## 15. 本节总结

这一节的完整训练逻辑是：

```text
准备训练数据
-> 划分训练集、验证集、测试集
-> 按 batch 喂给模型
-> 前向传播得到预测
-> 计算损失
-> 反向传播计算梯度
-> 优化器更新参数
-> 重复很多 iteration
-> 完成一个 epoch
-> 用验证集检查泛化能力
```

先记住几个概念：

- batch：一次喂给模型的一小批样本。
- iteration：一次参数更新。
- epoch：完整看完一遍训练集。
- 训练集：用来更新参数。
- 验证集：用来观察泛化能力。
- 测试集：最后评估模型。
- 过拟合：训练集好，新数据差。
- 欠拟合：训练集也学不好。

下一步就可以开始讲优化器：SGD、Momentum、Adam 到底分别在解决什么问题。